In [6]:
# Core imports
import numpy as np
import torch
import sys
from pathlib import Path
# Add project root to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import modules
from src.data.dataset import create_data_loaders
from src.models.loggptmodel import LogGPTModel
from src.engine.trainer import LogSeqTrainer
from src.utils.metrics import evaluate_model, print_metrics, save_experiment_results
from src.utils.data_loader import create_train_val_test_split, filter_normal_samples, load_loghub
from src.utils.visualizer import UniversalAnomalyVisualizer

In [7]:
# Define paths
DATA_DIR = '../data/hdfs/preprocessed'

In [8]:
# Load HDFS data using the utility function 
X, y, vocab = load_loghub(DATA_DIR)

vocab_size = len(vocab)
print(f"Vocab size: {vocab_size}")
print(f"Events: {sorted(vocab.keys())[:10]}")

INFO:src.utils.data_loader:Loading data from LogHub preprocessing: ../data/hdfs/preprocessed
INFO:src.utils.data_loader:Loaded data:
INFO:src.utils.data_loader:  - Sequences: (575061,)
INFO:src.utils.data_loader:  - Labels: (575061,)
INFO:src.utils.data_loader:  - Normal: 558223, Anomaly: 16838
INFO:src.utils.data_loader:  - Loaded vocabulary: 29 events


Vocab size: 29
Events: ['E1', 'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18']


In [9]:
# Split data (70/15/15)
splits = create_train_val_test_split(X, y, train_ratio=0.7, val_ratio=0.15, random_state=42)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = splits['train'], splits['val'], splits['test']

# Filter to keep only normal samples for semi-supervised training
X_train, y_train = filter_normal_samples(X_train, y_train, verbose=True)

INFO:src.utils.data_loader:Splitting data: train=0.7, val=0.15, test=0.15
INFO:src.utils.data_loader:Split complete:
INFO:src.utils.data_loader:  - Train: 402542 samples (11786 anomalies)
INFO:src.utils.data_loader:  - Val:   86259 samples (2526 anomalies)
INFO:src.utils.data_loader:  - Test:  86260 samples (2526 anomalies)
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:FILTERING TRAINING DATA FOR SEMI-SUPERVISED LEARNING
INFO:src.utils.data_loader:======================================================================
INFO:src.utils.data_loader:Original training size: 402542 samples
INFO:src.utils.data_loader:  Normal samples: 390,756 (97.07%)
INFO:src.utils.data_loader:  Anomaly samples: 11,786 (2.93%)
INFO:src.utils.data_loader:
Filtered training size: 390,756 samples (NORMAL ONLY)
INFO:src.utils.data_loader:Removed 11,786 anomalies from training set
INFO:src.utils.data_loader:✓ Training data is now pure no

In [10]:
# Convert strings to integers using the loaded vocab
def convert_to_ids(sequences, vocab):
    return [[vocab[event] for event in seq] for seq in sequences]

X_train_ids = convert_to_ids(X_train, vocab)
X_val_ids = convert_to_ids(X_val, vocab)
X_test_ids = convert_to_ids(X_test, vocab)

In [11]:
# Pad sequences (integers)
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = int(np.percentile([len(s) for s in X_train_ids], 95))

X_train_padded = pad_sequences(X_train_ids, maxlen=max_len, padding='post', value=0)
X_val_padded = pad_sequences(X_val_ids, maxlen=max_len, padding='post', value=0)
X_test_padded = pad_sequences(X_test_ids, maxlen=max_len, padding='post', value=0)

print(f"Max length: {max_len}")
print(f"Train shape: {X_train_padded.shape}")

Max length: 28
Train shape: (390756, 28)


In [12]:
# Create data loaders
batch_size = 64
train_loader, val_loader, test_loader = create_data_loaders(
    X_train_padded, y_train,
    X_val_padded, y_val,
    X_test_padded, y_test,
    batch_size=batch_size
)

In [13]:
#  Load the pre-computed embeddings
semantic_vectors = torch.load(f'{DATA_DIR}/semantic_embeddings.pt')
emb_dim = semantic_vectors.shape[1] 
print(f"Loaded semantic embeddings with dim: {emb_dim}")

Loaded semantic embeddings with dim: 384


In [14]:
model = LogGPTModel(
        vocab_size=vocab_size,
        embedding_dim=emb_dim,
        num_layers=6,
        num_heads=8,
        dropout=0.1,
        semantic_embeddings=semantic_vectors
    )

The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


LogGPT: Loaded semantic embeddings directly (384d).


In [16]:
# Print model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel Architecture:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Embedding dimension: {emb_dim}")
print(f"  Semantic embeddings: {'✓ Loaded' if semantic_vectors is not None else '✗ Random init'}")


Model Architecture:
  Total parameters: 10,855,296
  Trainable parameters: 10,855,296
  Embedding dimension: 384
  Semantic embeddings: ✓ Loaded


In [17]:
# Train
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
learning_rate = 0.001
patience = 20

print(f"Training on device: {device}")

trainer = LogSeqTrainer(model, device=device, learning_rate=learning_rate)

history = trainer.fit(
    train_loader, val_loader,
    num_epochs=50,
    early_stopping_patience=patience,
    print_every=5  # Print every 5 epochs
)

Training on device: mps


Training: 100%|██████████| 6106/6106 [10:23<00:00,  9.79it/s]



Epoch 1/50 - 663.33s
  Train Loss: 0.2432
  Val Loss:   0.2484
  ✓ New best model (val_loss: 0.2484)


Training: 100%|██████████| 6106/6106 [10:28<00:00,  9.71it/s]



Epoch 5/50 - 669.41s
  Train Loss: 0.2122
  Val Loss:   0.2366
  ✓ New best model (val_loss: 0.2366)


...


Epoch 30/50 - 654.18s
  Train Loss: 0.2092
  Val Loss:   0.2346
  No improvement (12/20)


Training: 100%|██████████| 6106/6106 [10:15<00:00,  9.92it/s]



Epoch 35/50 - 655.40s
  Train Loss: 0.2091
  Val Loss:   0.2342
  No improvement (17/20)


Training: 100%|██████████| 6106/6106 [10:30<00:00,  9.68it/s]



Early stopping triggered after 38 epochs

✓ Loaded best model (val_loss: 0.2335)
Total training time: 25182.61s


In [37]:
# Evaluate
# Capture predictions, true labels, AND anomaly scores from the loader
top_k = 5
predictions, true_labels, anomaly_scores = trainer.detect_anomalies(
    test_loader,
    top_k=top_k,
    return_scores=True
)

metrics = evaluate_model(predictions, true_labels) 
print_metrics(metrics)


Detecting anomalies: 100%|██████████| 1348/1348 [00:46<00:00, 29.20it/s]


EVALUATION METRICS
Accuracy:  0.9912 (99.12%)
Precision: 0.9570
Recall:    0.7312
F1-Score:  0.8290

Confusion Matrix:
              Predicted
              Normal  Anomaly
Actual Normal    83651       83
       Anomaly     679     1847


In [36]:
# Save experiment results
save_experiment_results(
    filepath="../results/hdfs_loggpt_results.json",
    dataset="HDFS",
    model_name="LogGPT",
    device=device,
    model=model,
    history=history,
    y_train=y_train,
    y_val=y_val,
    y_test=y_test,
    max_len=max_len,
    metrics=metrics,
    batch_size=batch_size,
    learning_rate=learning_rate,
    patience=patience,
    top_k=top_k,
    # LogAnomaly-specific params
    use_attention=True,
    n_heads=4
)

✓ Results saved to ../results/hdfs_loggpt_results.json


{'dataset': 'HDFS',
 'model': 'LogGPT',
 'device': 'mps',
 'architecture': {'vocab_size': 29,
  'embedding_dim': 384,
  'hidden_dim': None,
  'num_layers': None,
  'dropout': None,
  'use_attention': True,
  'num_attention_heads': 4},
 'training': {'num_epochs': 38,
  'batch_size': 64,
  'learning_rate': 0.001,
  'early_stopping_patience': 20,
  'best_val_loss': 0.23349721974054855,
  'training_time_seconds': 25182.61452484131},
 'data': {'train_size': 390756,
  'val_size': 86259,
  'test_size': 86260,
  'max_sequence_length': 28,
  'train_normal_only': True},
 'detection': {'top_k': 4, 'method': 'next_event_prediction'},
 'metrics': {'accuracy': 0.9903431486204498,
  'precision': 0.8740609809986744,
  'recall': 0.7830562153602534,
  'f1': 0.8260597201921069,
  'confusion_matrix': [[83449, 285], [548, 1978]]}}

In [20]:
# Save model checkpoint
save_dir = Path('../mdls')
save_dir.mkdir(parents=True, exist_ok=True)
save_path = save_dir / 'loggpt_checkpoint.pt'
trainer.save_model(str(save_path))

✓ Model saved to ../mdls/loggpt_checkpoint.pt
